# FastAPI Production Deployment

Covers health check endpoints, structured JSON logging, security headers middleware, performance tips (select specific columns, avoid N+1), and configuration best practices with Pydantic Settings.

In [ ]:
# !pip install fastapi httpx pydantic-settings

# --- Health check endpoint ---
from fastapi import FastAPI
from fastapi.testclient import TestClient
from datetime import datetime, timezone
import platform
import sys

app = FastAPI(title="Production App", version="1.0.0")

START_TIME = datetime.now(timezone.utc)

@app.get("/health", tags=["ops"])
def health_check():
    """Liveness probe — returns 200 if the app is running."""
    return {"status": "ok", "timestamp": datetime.now(timezone.utc).isoformat()}

@app.get("/health/ready", tags=["ops"])
def readiness_check():
    """Readiness probe — checks dependencies are available."""
    uptime_seconds = (datetime.now(timezone.utc) - START_TIME).total_seconds()
    checks = {
        "database": "ok",   # In real code: try a DB ping
        "cache": "ok",      # In real code: try a cache ping
    }
    all_ok = all(v == "ok" for v in checks.values())
    return {
        "status": "ready" if all_ok else "degraded",
        "uptime_seconds": round(uptime_seconds, 2),
        "python_version": sys.version.split()[0],
        "checks": checks
    }

client = TestClient(app)

r = client.get("/health")
print("Liveness:", r.json())

r = client.get("/health/ready")
print("Readiness:", r.json())

In [ ]:
# --- Structured JSON logging ---
import logging
import json
import sys
from datetime import datetime, timezone
from fastapi import FastAPI, Request
from fastapi.testclient import TestClient
from starlette.middleware.base import BaseHTTPMiddleware

class JSONFormatter(logging.Formatter):
    """Formats log records as JSON for structured logging."""
    def format(self, record: logging.LogRecord) -> str:
        log_entry = {
            "timestamp": datetime.now(timezone.utc).isoformat(),
            "level": record.levelname,
            "logger": record.name,
            "message": record.getMessage(),
        }
        if record.exc_info:
            log_entry["exception"] = self.formatException(record.exc_info)
        # Include any extra fields passed to the logger
        for key, val in record.__dict__.items():
            if key not in logging.LogRecord.__dict__ and not key.startswith("_"):
                log_entry[key] = val
        return json.dumps(log_entry)

# Configure logger
logger = logging.getLogger("app")
logger.setLevel(logging.DEBUG)
handler = logging.StreamHandler(sys.stdout)
handler.setFormatter(JSONFormatter())
logger.handlers = [handler]
logger.propagate = False

class AccessLogMiddleware(BaseHTTPMiddleware):
    async def dispatch(self, request: Request, call_next):
        import time
        start = time.perf_counter()
        response = await call_next(request)
        duration_ms = round((time.perf_counter() - start) * 1000, 2)
        logger.info(
            "HTTP request",
            extra={
                "method": request.method,
                "path": request.url.path,
                "status_code": response.status_code,
                "duration_ms": duration_ms
            }
        )
        return response

app2 = FastAPI()
app2.add_middleware(AccessLogMiddleware)

@app2.get("/data")
def get_data():
    logger.info("Fetching data", extra={"record_count": 42})
    return {"data": "some value"}

client2 = TestClient(app2)
print("Making request (JSON logs below):")
r = client2.get("/data")
print("Response:", r.json())

In [ ]:
# --- Security headers middleware ---
from fastapi import FastAPI
from fastapi.testclient import TestClient
from starlette.middleware.base import BaseHTTPMiddleware
from starlette.requests import Request

class SecurityHeadersMiddleware(BaseHTTPMiddleware):
    """Adds OWASP-recommended security headers to every response."""
    async def dispatch(self, request: Request, call_next):
        response = await call_next(request)
        response.headers["X-Content-Type-Options"] = "nosniff"
        response.headers["X-Frame-Options"] = "DENY"
        response.headers["X-XSS-Protection"] = "1; mode=block"
        response.headers["Strict-Transport-Security"] = "max-age=31536000; includeSubDomains"
        response.headers["Content-Security-Policy"] = "default-src 'self'"
        response.headers["Referrer-Policy"] = "strict-origin-when-cross-origin"
        response.headers["Permissions-Policy"] = "geolocation=(), microphone=()"
        # Remove server fingerprinting
        response.headers.pop("server", None)
        return response

app3 = FastAPI()
app3.add_middleware(SecurityHeadersMiddleware)

@app3.get("/")
def root():
    return {"message": "Secure endpoint"}

client3 = TestClient(app3)
r = client3.get("/")

security_headers = [
    "x-content-type-options",
    "x-frame-options",
    "x-xss-protection",
    "strict-transport-security",
    "content-security-policy",
    "referrer-policy",
    "permissions-policy",
]

print("Security headers present:")
for h in security_headers:
    val = r.headers.get(h, "MISSING")
    status = "✓" if val != "MISSING" else "✗"
    print(f"  {status} {h}: {val}")

In [ ]:
# --- Performance tips: select specific columns, avoid N+1 ---
# Demonstrates the patterns using in-memory data (no real DB needed)

# Tip 1: Select only needed columns (projection)
USERS_FULL = [
    {"id": i, "name": f"User {i}", "email": f"user{i}@example.com",
     "password_hash": "$2b$...", "internal_notes": "sensitive", "age": 20 + i}
    for i in range(1, 6)
]

def project(record: dict, fields: list[str]) -> dict:
    return {k: v for k, v in record.items() if k in fields}

# BAD: returns all fields including sensitive ones
bad_result = USERS_FULL[:2]
print("BAD (all fields):", bad_result[0].keys())

# GOOD: only return what the client needs
good_result = [project(u, ["id", "name", "email"]) for u in USERS_FULL[:2]]
print("GOOD (projected):", good_result)

# Tip 2: Avoid N+1 — batch load related data
ORDERS = [
    {"id": 1, "user_id": 1, "total": 29.99},
    {"id": 2, "user_id": 2, "total": 49.99},
    {"id": 3, "user_id": 1, "total": 9.99},
]

# BAD N+1: for each order, look up user individually
def get_user_by_id_slow(user_id: int):
    # Simulates a separate DB query per call
    return next((u for u in USERS_FULL if u["id"] == user_id), None)

print("\nBAD N+1 pattern (1 query per order):")
for order in ORDERS:
    user = get_user_by_id_slow(order["user_id"])  # N separate queries!
    print(f"  Order {order['id']} -> {user['name']}")

# GOOD: batch load all needed users in one query
print("\nGOOD batch load (1 query total):")
needed_user_ids = {o["user_id"] for o in ORDERS}
users_map = {u["id"]: u for u in USERS_FULL if u["id"] in needed_user_ids}  # 1 query
for order in ORDERS:
    user = users_map[order["user_id"]]
    print(f"  Order {order['id']} -> {user['name']}")

In [ ]:
# --- Configuration best practices with Pydantic Settings ---
# !pip install pydantic-settings

from pydantic_settings import BaseSettings, SettingsConfigDict
from pydantic import Field
from functools import lru_cache
from fastapi import FastAPI, Depends
from fastapi.testclient import TestClient

class Settings(BaseSettings):
    # App settings
    app_name: str = Field(default="MyFastAPIApp", description="Application name")
    debug: bool = Field(default=False, description="Enable debug mode")
    version: str = Field(default="1.0.0")

    # Database
    database_url: str = Field(default="sqlite+aiosqlite:///./app.db")
    db_pool_size: int = Field(default=5, ge=1, le=50)

    # Security
    secret_key: str = Field(default="change-me-in-production")
    access_token_expire_minutes: int = Field(default=30, ge=1)

    # CORS
    allowed_origins: list[str] = Field(default=["http://localhost:3000"])

    model_config = SettingsConfigDict(
        env_file=".env",          # Load from .env file
        env_file_encoding="utf-8",
        case_sensitive=False,     # DATABASE_URL or database_url both work
        extra="ignore"            # Ignore unknown env vars
    )

@lru_cache  # Cache settings — only reads env once
def get_settings() -> Settings:
    return Settings()

app4 = FastAPI()

@app4.get("/config")
def show_config(settings: Settings = Depends(get_settings)):
    # Never expose secrets in real code!
    return {
        "app_name": settings.app_name,
        "version": settings.version,
        "debug": settings.debug,
        "db_pool_size": settings.db_pool_size,
        "token_expire_minutes": settings.access_token_expire_minutes,
        "allowed_origins": settings.allowed_origins,
        "secret_key_set": settings.secret_key != "change-me-in-production"
    }

client4 = TestClient(app4)
r = client4.get("/config")
print("Config:", r.json())

# Override via dependency_overrides for testing
def get_test_settings():
    return Settings(app_name="TestApp", debug=True, secret_key="test-secret-key")

app4.dependency_overrides[get_settings] = get_test_settings
r = client4.get("/config")
print("Test config:", r.json())
app4.dependency_overrides.clear()